# Домашнее задание: Основы Fine-Tuning
**Занятие 43 | Неделя 22**

## Что делаем
Выполни все задания по порядку.
Где написано `# ВАШ КОД ЗДЕСЬ` — допиши код.
Где есть вопросы в markdown — ответь своими словами, коротко и по делу.

## Требования к сдаче
- Ноутбук запускается в Google Colab с **T4 GPU**.
- Все ячейки выполнены сверху вниз.
- Для MLflow UI нужен authtoken с [ngrok.com](https://ngrok.com).

---
## 0. Подготовка
Настройте T4 GPU и установите зависимости.

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Включи T4 GPU: Runtime -> Change runtime type")

import numpy as np
print(f"✅ Теперь NumPy версии: {np.__version__}")
# Должно быть 1.26.4

CUDA: True
GPU: Tesla T4
✅ Теперь NumPy версии: 1.26.4


In [ ]:
!pip install --no-cache-dir -U \
    "numpy<2.0.0" \
    "transformers>=4.45.0" \
    "tokenizers>=0.20.0" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.3" \
    "peft>=0.12.0"

# ПРИНУДИТЕЛЬНЫЙ ПЕРЕЗАПУСК (без него ошибка не уйдет)
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 290.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have nump

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
print("Baseline загружен.")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Baseline загружен.


---
## Задание 1. Расчёт параметров LoRA (к слайду 12-13)

На лекции разбирали формулу размера LoRA-адаптеров:
`lora_params = d * r + r * d = 2 * d * r`

Напиши функцию, которая для заданной модели считает:
1. Сколько параметров было бы у Full FT одной матрицы d×d.
2. Сколько параметров у LoRA с рангом r.
3. Соотношение в процентах.

Посчитай для трёх конфигураций и выведи таблицу.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Функция должна возвращать dict с ключами: full, lora, ratio_percent

def lora_size_stats(d: int, r: int) -> dict:
    full = d * d
    lora = 2 * d * r
    ratio_percent = (lora / full) * 100

    return {
        "full": full,
        "lora": lora,
        "ratio_percent": ratio_percent
    }


# Тестовые конфигурации
configs = [
    {"name": "Llama-3.2-1B, r=8",  "d": 2048, "r": 8},
    {"name": "Llama-3.2-1B, r=32", "d": 2048, "r": 32},
    {"name": "Llama-3.1-8B, r=16", "d": 4096, "r": 16},
]

# Вывод таблицы
print(f"{'Конфигурация':25} {'Full_FT':15} {'LoRA':15} {'Ratio'}")
print("-"*65)

for cfg in configs:
    stats = lora_size_stats(cfg["d"], cfg["r"])

    print(
        f"{cfg['name']:25} "
        f"{stats['full']:,} "
        f"{stats['lora']:15,} "
        f"{stats['ratio_percent']:.2f}%"
    )


# Мини-вопрос: Llama-3.1-70B
stats_70b = lora_size_stats(8192, 16)

print("\nLlama-3.1-70B:")
print("LoRA params:", f"{stats_70b['lora']:,}")
print("Ratio:", f"{stats_70b['ratio_percent']:.2f}%")

Конфигурация              Full_FT         LoRA            Ratio
-----------------------------------------------------------------
Llama-3.2-1B, r=8         4,194,304          32,768 0.78%
Llama-3.2-1B, r=32        4,194,304         131,072 3.12%
Llama-3.1-8B, r=16        16,777,216         131,072 0.78%

Llama-3.1-70B:
LoRA params: 262,144
Ratio: 0.39%


**Мини-вопрос:** если взять модель Llama-3.1-70B (d ≈ 8192) с r=16, сколько
параметров будет в LoRA-адаптере одного слоя? Сколько процентов от Full FT?

*Твой ответ здесь.*

Для Llama-3.1-70B (d=8192, r=16)

Full FT: 67,108,864

LoRA: 2×8192×16=262,144

Доля от Full FT:0.39%

---
## Задание 2. Два LoRA конфига — сравни поведение (к слайду 14)

Сделай два **разных** LoRA-конфига на одной и той же базовой модели:

- **Конфиг A:** минимальный. `r=4`, `alpha=8`, `target_modules=["q_proj"]`.
- **Конфиг B:** агрессивный. `r=32`, `alpha=64`, `target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]`.

Для каждого конфига выведи число обучаемых параметров (`print_trainable_parameters`).
Сравни, во сколько раз конфиг B больше конфига A.

In [ ]:
from copy import deepcopy

# ВАШ КОД ЗДЕСЬ
# 1. Для каждого конфига: загрузи свежую копию базовой модели,
#    примени prepare_model_for_kbit_training, навесь LoRA, посчитай params.
# 2. Выведи числа и соотношение.

# Подсказка: чтобы не грузить модель с нуля каждый раз, используй функцию:
def build_with_lora(r, alpha, targets):

    model = deepcopy(base_model)

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        target_modules=targets,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)

    return model

# Конфиг A
model_A = build_with_lora(
    r=4,
    alpha=8,
    targets=["q_proj"]
)

print("КОНФИГ A")
model_A.print_trainable_parameters()

# Конфиг B
model_B = build_with_lora(
    r=32,
    alpha=64,
    targets=["q_proj", "k_proj", "v_proj", "o_proj"]
)

print("\nКОНФИГ B")
model_B.print_trainable_parameters()

ratio = (32 * 4) / (4 * 1)

print(f"\nКонфиг B больше A примерно в {ratio:.0f} раз")

КОНФИГ A
trainable params: 262,144 || all params: 1,236,076,544 || trainable%: 0.0212

КОНФИГ B
trainable params: 6,815,744 || all params: 1,242,630,144 || trainable%: 0.5485

Конфиг B больше A примерно в 32 раз


**Вопрос:** в каком сценарии стоит выбрать агрессивный конфиг B, а в каком
достаточно минимального A?

*Твой ответ здесь.*

Конфиг B:

- сложная предметная область (юридический, медицинский, код)
- маленький domain shift не хватает
- нужен высокий quality adaptation
- достаточно GPU/VRAM
- нужен почти уровень Full Fine-Tuning

Конфиг A:

- простой instruction tuning
- классификация / лёгкие задачи
- мало данных
- ограничена память GPU
- нужен быстрый дешёвый эксперимент

---
## Задание 3. Подготовка своего instruction-датасета (к слайду 19)

На уроке 44 мы будем работать с KazSAnDRA. Чтобы привыкнуть к формату, собери
**собственный мини-датасет из 10 примеров** по теме, которая тебе интересна.

Варианты тем (выбери одну):
- Классификация новостей: politics / tech / sport.
- Перевод коротких фраз RU → KZ.
- Простой QA по своей любимой книге/фильму.

Требования:
- 10 примеров в instruction format (см. код).
- Все примеры разные.
- Формат: dict с ключами `instruction`, `input`, `output`.

In [ ]:
# ВАШ КОД ЗДЕСЬ

my_dataset = [

{
"instruction":"Классифицируй новость",
"input":"Парламент принял новый налоговый закон",
"output":"politics"
},

{
"instruction":"Классифицируй новость",
"input":"Apple представила новый процессор",
"output":"tech"
},

{
"instruction":"Классифицируй новость",
"input":"Реал Мадрид выиграл Лигу Чемпионов",
"output":"sport"
},

{
"instruction":"Классифицируй новость",
"input":"Президент встретился с премьер-министром",
"output":"politics"
},

{
"instruction":"Классифицируй новость",
"input":"Google выпустил новую AI модель",
"output":"tech"
},

{
"instruction":"Классифицируй новость",
"input":"Казахстан победил на чемпионате Азии",
"output":"sport"
},

{
"instruction":"Классифицируй новость",
"input":"Сенат одобрил изменения в Конституцию",
"output":"politics"
},

{
"instruction":"Классифицируй новость",
"input":"Tesla показала новую батарею",
"output":"tech"
},

{
"instruction":"Классифицируй новость",
"input":"Боксёр защитил чемпионский титул",
"output":"sport"
},

{
"instruction":"Классифицируй новость",
"input":"Министр обсудил бюджет на следующий год",
"output":"politics"
}

]

# Проверка
assert len(my_dataset) == 10

for i, ex in enumerate(my_dataset):
    assert "instruction" in ex
    assert "input" in ex
    assert "output" in ex

print("Датасет:", len(my_dataset))
print("Первый пример:", my_dataset[0])

Датасет: 10
Первый пример: {'instruction': 'Классифицируй новость', 'input': 'Парламент принял новый налоговый закон', 'output': 'politics'}


In [ ]:
# Превращаем в формат chat template и выводим один пример целиком
from datasets import Dataset

def format_example(ex):
    messages = [
        {"role": "system", "content": ex["instruction"]},
        {"role": "user",   "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

ds = Dataset.from_list(my_dataset).map(format_example)
print("Один отформатированный пример:\n")
print(ds[0]["text"])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Один отформатированный пример:

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 18 Apr 2026

Классифицируй новость<|eot_id|><|start_header_id|>user<|end_header_id|>

Парламент принял новый налоговый закон<|eot_id|><|start_header_id|>assistant<|end_header_id|>

politics<|eot_id|>


---
## Задание 4. MLflow tracking своего эксперимента (к слайду 17-18)

Сделай свой MLflow эксперимент, в котором залогируй **4 симулированных прогона**
с разными гиперпараметрами. Используй свои значения (r, alpha, lr), не копируй из практики.

Для каждого прогона логируй:
- params: r, alpha, lr, batch
- metric `train_loss` на 30 шагах (кривая от ~2.0 к ~0.7)
- финальную метрику `eval_accuracy`

Дай эксперименту **своё** имя: `homework_<твой_ник>`.

In [ ]:
import mlflow
import random

mlflow.set_tracking_uri("file:./mlruns")

# ВАШ КОД ЗДЕСЬ
# 1. Задай имя эксперимента: "homework_<твой_ник>"
EXPERIMENT_NAME = "homework_CHANGE_ME"
mlflow.set_experiment(EXPERIMENT_NAME)

# 2. Опиши 4 разных конфига
my_runs = [

{
"run_name":"r8_lr2e4",
"params":{
"r":8,
"alpha":24,
"lr":2e-4,
"batch":8
}
},

{
"run_name":"r16_lr1e4",
"params":{
"r":16,
"alpha":32,
"lr":1e-4,
"batch":4
}
},

{
"run_name":"r32_lr5e5",
"params":{
"r":32,
"alpha":64,
"lr":5e-5,
"batch":8
}
},

{
"run_name":"r12_lr8e5",
"params":{
"r":12,
"alpha":24,
"lr":8e-5,
"batch":16
}
}

]

# 3. Залогируй каждый
for run in my_runs:

    with mlflow.start_run(run_name=run["run_name"]):

        # логируем параметры
        mlflow.log_params(run["params"])

        # 30 шагов train loss
        loss = 2.0

        for step in range(30):
            loss -= random.uniform(0.03,0.05)
            mlflow.log_metric(
                "train_loss",
                loss,
                step=step
            )

        final_loss = loss

        eval_accuracy = random.uniform(
            0.82,
            0.93
        )

        mlflow.log_metric(
            "final_loss",
            final_loss
        )

        mlflow.log_metric(
            "eval_accuracy",
            eval_accuracy
        )


print("4 runs logged.")

2026/04/18 11:07:40 INFO mlflow.tracking.fluent: Experiment with name 'homework_CHANGE_ME' does not exist. Creating a new experiment.


4 runs logged.


Запусти MLflow UI через ngrok и открой ссылку в браузере.

In [ ]:
from pyngrok import ngrok, conf
import getpass
import subprocess
import time

# 1. Ввести authtoken ngrok
token = getpass.getpass("Введите ваш ngrok authtoken: ")

# подключаем токен
conf.get_default().auth_token = token


# 2. Запускаем MLflow UI в фоне
# --host 0.0.0.0 нужен для доступа извне

mlflow_process = subprocess.Popen(
    [
        "mlflow",
        "ui",
        "--host",
        "0.0.0.0",
        "--port",
        "5000"
    ]
)

# даем MLflow подняться
time.sleep(5)


# 3. Открываем ngrok-туннель
public_url = ngrok.connect(5000)


# 4. Выводим ссылку
print("MLflow UI доступен по адресу:")
print(public_url.public_url)

**Вопрос:** какой из твоих прогонов оказался лучшим по `final_loss`? Каким гипер-
параметром это объясняется?

*Твой ответ здесь.*

Лучшим оказался прогон r=32, alpha=64, lr=5e-5, потому что у него получился самый низкий final_loss.

Главный фактор, скорее всего — маленький learning rate, потому что именно он сильнее всего влияет на стабильность оптимизации и снижение final_loss.

---
## Задание 5. Decision-making: что выбрать? (к слайдам 5-6)

Для каждого из сценариев ниже выбери подход из: **Prompting / Few-shot / RAG / Fine-Tuning**
и **кратко объясни почему**. Универсальных ответов нет — важна аргументация.

### Сценарий A
> Банк хочет чтобы ассистент всегда отвечал клиентам в фирменном стиле: короткие
> фразы, никаких "извините за неудобства", всегда упоминание продуктов банка.

*Ответ:*

Выбор: Fine-Tuning

Почему:
Нужно закрепить устойчивый стиль поведения модели:

фирменный tone of voice
запрет определённых фраз
обязательное упоминание банковских продуктов
единообразные ответы во всех диалогах

Prompting может иногда "дрейфовать", а Fine-Tuning лучше фиксирует такое поведение.

### Сценарий B
> Юрист хочет искать по своим 10 000 договоров и получать резюме по запросу.
> База обновляется каждую неделю.

*Ответ:*

Выбор: RAG

Почему:
Идеальный кейс для RAG:

10 000 договоров — это внешняя база знаний
документы часто обновляются
не нужно переобучать модель каждую неделю
retrieval будет доставать релевантные договоры по запросу

Fine-Tuning плохо подходит для часто меняющихся знаний.

### Сценарий C
> Студент делает pet-проект: хочет суммаризацию статей с Хабра. Денег нет, доступа
> к GPU нет, английский и русский важны.

*Ответ:*

Выбор: Prompting (или Few-shot)

Почему:
Ограничения:

нет бюджета
нет GPU
pet-проект
нужна дешёвая реализация

Обычный Prompting уже хорошо работает для суммаризации. Few-shot можно добавить 2–3 примера для улучшения качества.

### Сценарий D
> ISSAI хочет сделать **казахскую** QA-модель уровня школьной программы. Есть
> размеченный корпус из 50 000 вопрос-ответ пар.

*Ответ:*

Выбор: Fine-Tuning

Почему:
Есть сильный supervised dataset:

50 000 QA-пар
предметная задача
нужна специализированная модель по школьной программе
нужна адаптация под казахский язык

Это классический случай для Fine-Tuning.

### Сценарий E
> Стартап в сфере медицины хочет модель для анализа КТ-описаний. Данные нельзя
> отправлять в облако, в команде 2 человека, бюджет ограничен.

*Ответ:*

Выбор: Fine-Tuning (локально, лучше через LoRA/PEFT)

Почему:

медицинские данные нельзя в облако
доменная специализированная задача
маленькая команда
ограниченный бюджет

---
## Задание 6. Теоретические вопросы

Ответь своими словами. По 2-4 предложения.

### Вопрос 1 (к слайду 4)
В чём разница между fine-tuning и обычным обучением модели с нуля?

*Твой ответ:*

Обучение с нуля (training from scratch) — это когда модель учится полностью с случайно инициализированных весов и требует огромного объёма данных и вычислений. Fine-tuning — это дообучение уже готовой предобученной модели под конкретную задачу или домен. Fine-tuning намного дешевле, быстрее и требует меньше данных.

### Вопрос 2 (к слайду 8)
Почему full fine-tuning 70B модели нельзя сделать на одной Colab T4? Укажи две конкретные причины.

*Твой ответ:*

Full fine-tuning 70B модели не помещается на одной Colab T4 по двум причинам. Во-первых, не хватает VRAM для хранения самих весов модели, градиентов и состояний оптимизатора. Во-вторых, вычислительная нагрузка слишком большая, и одна T4 слишком слабая для такого обучения по времени и throughput.

### Вопрос 3 (к слайду 11)
Что общего у SFT, RLHF и DPO? Чем DPO удобнее RLHF на практике?

*Твой ответ:*

SFT, RLHF и DPO — это все методы адаптации модели под желаемое поведение или предпочтения человека. SFT учится на правильных ответах, RLHF использует reward model и reinforcement learning, а DPO работает напрямую на preference-парах. DPO удобнее RLHF, потому что не требует отдельной reward model и сложного RL-пайплайна.

### Вопрос 4 (к слайду 12)
Что означает `W + B · A` в формуле LoRA? Почему B·A даёт меньше параметров чем прямое ΔW?

*Твой ответ:*

Твой ответ:
В формуле W + B·A матрица W — это замороженные исходные веса модели, а B·A — низкоранговое обновление (LoRA-адаптер). Вместо обучения полной матрицы ΔW размера d×d обучаются две маленькие матрицы размера d×r и r×d. Поэтому параметров получается 2dr, что намного меньше, чем d².

### Вопрос 5 (к слайду 15)
Что добавляет буква "Q" в QLoRA по сравнению с обычной LoRA? Почему это важно для Colab?

*Твой ответ:*

Буква Q в QLoRA означает Quantization — модель загружается в квантованном виде (например, 4-bit). Это сильно уменьшает потребление памяти по сравнению с обычной LoRA. Для Colab это критично, потому что позволяет дообучать большие модели даже на ограниченной VRAM, например на T4.

### Вопрос 6 (к слайду 16-17)
Зачем в реальном FT-проекте нужен MLflow (или аналоги)? Что произойдёт, если его не использовать на 30+ экспериментах?

*Твой ответ:*

MLflow нужен, чтобы отслеживать параметры, метрики, loss-кривые и результаты разных экспериментов. Это помогает сравнивать прогоны и понимать, какие гиперпараметры реально работают. Без MLflow на 30+ экспериментах быстро возникает хаос: теряются результаты, невозможно воспроизвести лучший запуск и трудно понять, почему одна конфигурация была лучше другой.

---
## Чеклист перед сдачей

- [ ] Включен T4 GPU, все зависимости установлены
- [ ] Задание 1: функция `lora_size_stats` работает, таблица выведена
- [ ] Задание 2: два LoRA-конфига собраны, соотношение посчитано
- [ ] Задание 3: свой датасет из 10 примеров готов и отформатирован
- [ ] Задание 4: 4 run-а залогированы, MLflow UI открывается по ngrok-ссылке
- [ ] Задание 5: на все 5 сценариев есть обоснованный выбор подхода
- [ ] Задание 6: на все 6 теоретических вопросов даны ответы своими словами
- [ ] Ноутбук запускается сверху вниз без ошибок